# RAGAS — RAG Evaluation Practical

This notebook uses the **current collections-based RAGAS API** recommended in the stable docs for new projects.

We will build a mini RAG system and evaluate it with:

- Context Precision
- Context Recall
- Faithfulness
- Answer Relevancy
- Factual Correctness

Then we will run the same metrics over a small evaluation dataset and build a report.

## 0. Installation

In [ ]:
%pip install -U ragas openai langchain langchain-openai pandas

## 1. Imports

In [ ]:
import os
import getpass
import pandas as pd

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory
from ragas.metrics.collections import (
    ContextPrecision,
    ContextRecall,
    Faithfulness,
    AnswerRelevancy,
    FactualCorrectness,
)

print('Imports successful.')

## 2. API Key and Models

In [ ]:
if not os.getenv('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')

RAG_MODEL = os.getenv('RAG_MODEL', 'gpt-4o-mini')
RAG_EMBEDDING_MODEL = os.getenv('RAG_EMBEDDING_MODEL', 'text-embedding-3-small')
RAGAS_JUDGE_MODEL = os.getenv('RAGAS_JUDGE_MODEL', 'gpt-4o-mini')
RAGAS_EMBEDDING_MODEL = os.getenv('RAGAS_EMBEDDING_MODEL', 'text-embedding-3-small')

print('RAG model:', RAG_MODEL)
print('RAGAS judge:', RAGAS_JUDGE_MODEL)

## 3. Mini Knowledge Base

In [ ]:
documents = [
    Document(page_content='Full-time employees receive 24 paid leaves per calendar year.', metadata={'doc_id':'leave_policy'}),
    Document(page_content='Employees are allowed to work from home for a maximum of 2 days per week.', metadata={'doc_id':'remote_policy'}),
    Document(page_content='Employees can claim up to ₹3000 per month for internet reimbursement.', metadata={'doc_id':'internet_policy'}),
    Document(page_content='The standard probation period for new employees is 6 months.', metadata={'doc_id':'probation_policy'}),
    Document(page_content='Employees receive ₹1000 per month as mobile reimbursement.', metadata={'doc_id':'mobile_policy'}),
    Document(page_content="Medical insurance coverage begins from the employee's date of joining.", metadata={'doc_id':'insurance_policy'}),
]

print('Documents:', len(documents))

## 4. Vector Store and Retriever

In [ ]:
rag_embeddings = OpenAIEmbeddings(model=RAG_EMBEDDING_MODEL)
vector_store = InMemoryVectorStore(embedding=rag_embeddings)
vector_store.add_documents(documents)

TOP_K = 3
retriever = vector_store.as_retriever(search_kwargs={'k': TOP_K})

print('Retriever ready. TOP_K =', TOP_K)

## 5. Generator LLM

In [ ]:
rag_llm = ChatOpenAI(
    model=RAG_MODEL,
    temperature=0,
)

## 6. Complete RAG Pipeline

In [ ]:
def rag_pipeline(query: str) -> dict:
    retrieved_docs = retriever.invoke(query)

    retrieved_contexts = [doc.page_content for doc in retrieved_docs]
    retrieved_doc_ids = [doc.metadata.get('doc_id') for doc in retrieved_docs]
    context = '\n\n'.join(retrieved_contexts)

    prompt = f"""
You are an HR policy assistant.
Answer the question using ONLY the supplied context.
If the answer is missing, say: "I don't know based on the provided context."
Do not invent policy details.

CONTEXT:
{context}

QUESTION:
{query}
"""

    response = rag_llm.invoke(prompt)

    return {
        'response': response.content,
        'retrieved_contexts': retrieved_contexts,
        'retrieved_doc_ids': retrieved_doc_ids,
    }

## 7. Sanity Check the RAG

In [ ]:
sample_query = 'What is the monthly internet reimbursement limit?'
sample = rag_pipeline(sample_query)

print('RESPONSE:')
print(sample['response'])
print('\nDOC IDS:', sample['retrieved_doc_ids'])
print('\nCONTEXTS:')
for i, c in enumerate(sample['retrieved_contexts'], 1):
    print(f'{i}. {c}')

# PART 2 — Configure RAGAS

RAGAS can use a separate evaluator LLM and embedding model. The current stable docs expose `llm_factory()` and `embedding_factory()` for this.

In [ ]:
openai_client = OpenAI(
    api_key=os.environ['OPENAI_API_KEY'],
    timeout=60.0,
    max_retries=3,
)

evaluator_llm = llm_factory(
    RAGAS_JUDGE_MODEL,
    client=openai_client,
)

evaluator_embeddings = embedding_factory(
    'openai',
    model=RAGAS_EMBEDDING_MODEL,
    client=openai_client,
)

print('RAGAS evaluator models created.')

## 8. Create Metrics

In [ ]:
context_precision = ContextPrecision(llm=evaluator_llm)
context_recall = ContextRecall(llm=evaluator_llm)
faithfulness = Faithfulness(llm=evaluator_llm)
answer_relevancy = AnswerRelevancy(
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)
factual_correctness = FactualCorrectness(
    llm=evaluator_llm,
    mode='f1',
)

print('Metrics ready.')

## Metric Mental Model

| Metric | Main Question | Component |
|---|---|---|
| Context Precision | Are relevant chunks ranked high? | Retriever / reranker |
| Context Recall | Was required information retrieved? | Retriever |
| Faithfulness | Is the answer grounded in retrieved context? | Generator |
| Answer Relevancy | Does the response address the question? | Generator |
| Factual Correctness | Does response align with reference facts? | Final answer |

# PART 3 — Evaluate One RAG Response

In [ ]:
question = 'What is the monthly internet reimbursement limit?'
reference = 'Employees can claim up to ₹3000 per month for internet reimbursement.'

run = rag_pipeline(question)
response = run['response']
retrieved_contexts = run['retrieved_contexts']

print('Question:', question)
print('Reference:', reference)
print('Response:', response)
print('Retrieved contexts:', retrieved_contexts)

## 9. Context Precision

In [ ]:
cp = context_precision.score(
    user_input=question,
    reference=reference,
    retrieved_contexts=retrieved_contexts,
)
print('Context Precision:', cp.value)

## 10. Context Recall

In [ ]:
cr = context_recall.score(
    user_input=question,
    reference=reference,
    retrieved_contexts=retrieved_contexts,
)
print('Context Recall:', cr.value)

## 11. Faithfulness

In [ ]:
faith = faithfulness.score(
    user_input=question,
    response=response,
    retrieved_contexts=retrieved_contexts,
)
print('Faithfulness:', faith.value)

## 12. Answer Relevancy

In [ ]:
rel = answer_relevancy.score(
    user_input=question,
    response=response,
)
print('Answer Relevancy:', rel.value)

## 13. Factual Correctness

In [ ]:
fact = factual_correctness.score(
    response=response,
    reference=reference,
)
print('Factual Correctness:', fact.value)

## 14. One-Question Summary

In [ ]:
pd.DataFrame([{
    'question': question,
    'response': response,
    'context_precision': cp.value,
    'context_recall': cr.value,
    'faithfulness': faith.value,
    'answer_relevancy': rel.value,
    'factual_correctness': fact.value,
}])

# PART 4 — Evaluation Dataset

In [ ]:
evaluation_dataset = [
    {
        'question':'How many paid leaves does a full-time employee receive?',
        'reference':'A full-time employee receives 24 paid leaves per calendar year.',
    },
    {
        'question':'How many work-from-home days are allowed per week?',
        'reference':'Employees can work from home for a maximum of 2 days per week.',
    },
    {
        'question':'What is the monthly internet reimbursement limit?',
        'reference':'Employees can claim up to ₹3000 per month for internet reimbursement.',
    },
    {
        'question':'What is the standard probation period?',
        'reference':'The standard probation period for new employees is 6 months.',
    },
    {
        'question':'When does medical insurance coverage begin?',
        'reference':"Medical insurance coverage begins from the employee's date of joining.",
    },
]

pd.DataFrame(evaluation_dataset)

# PART 5 — Full RAGAS Evaluation

This deliberately runs sequentially so it is easier to debug and less aggressive on API rate limits.

In [ ]:
evaluation_results = []

for i, sample in enumerate(evaluation_dataset, 1):
    print(f'Evaluating {i}/{len(evaluation_dataset)}')

    q = sample['question']
    ref = sample['reference']
    rag = rag_pipeline(q)
    resp = rag['response']
    contexts = rag['retrieved_contexts']

    cp = context_precision.score(
        user_input=q, reference=ref, retrieved_contexts=contexts
    )
    cr = context_recall.score(
        user_input=q, reference=ref, retrieved_contexts=contexts
    )
    faith = faithfulness.score(
        user_input=q, response=resp, retrieved_contexts=contexts
    )
    rel = answer_relevancy.score(
        user_input=q, response=resp
    )
    fact = factual_correctness.score(
        response=resp, reference=ref
    )

    evaluation_results.append({
        'question': q,
        'response': resp,
        'reference': ref,
        'retrieved_doc_ids': rag['retrieved_doc_ids'],
        'context_precision': cp.value,
        'context_recall': cr.value,
        'faithfulness': faith.value,
        'answer_relevancy': rel.value,
        'factual_correctness': fact.value,
    })

print('Evaluation complete.')

## 15. Detailed Report

In [ ]:
results_df = pd.DataFrame(evaluation_results)
results_df

## 16. Compact Score Table

In [ ]:
score_columns = [
    'question',
    'context_precision',
    'context_recall',
    'faithfulness',
    'answer_relevancy',
    'factual_correctness',
]
results_df[score_columns]

## 17. Aggregate Report

In [ ]:
metric_columns = [
    'context_precision',
    'context_recall',
    'faithfulness',
    'answer_relevancy',
    'factual_correctness',
]

aggregate_report = results_df[metric_columns].mean().to_frame('mean_score')
aggregate_report

## 18. Diagnostic Helper

In [ ]:
def diagnose(row, threshold=0.70):
    issues = []
    if row['context_precision'] < threshold:
        issues.append('Retriever ranking/noise issue')
    if row['context_recall'] < threshold:
        issues.append('Retriever missing information')
    if row['faithfulness'] < threshold:
        issues.append('Generator grounding issue')
    if row['answer_relevancy'] < threshold:
        issues.append('Answer not sufficiently relevant')
    if row['factual_correctness'] < threshold:
        issues.append('Answer/reference factual mismatch')
    return ' | '.join(issues) if issues else 'No obvious issue at this threshold'

diagnostic_df = results_df.copy()
diagnostic_df['diagnosis'] = diagnostic_df.apply(diagnose, axis=1)
diagnostic_df[['question'] + metric_columns + ['diagnosis']]

## 19. Export Results

In [ ]:
results_df.to_csv('ragas_rag_detailed_results.csv', index=False)
aggregate_report.to_csv('ragas_rag_aggregate_report.csv')
diagnostic_df.to_csv('ragas_rag_diagnostics.csv', index=False)
print('CSV files exported.')

# Use With Your Existing RAG

Your own RAG only needs to expose the generated `response` and the actual `retrieved_contexts`. Then use the same scorer calls.

## Current API note

This notebook intentionally uses:

```python
from ragas.metrics.collections import ...
metric.score(...)
```

The stable docs mark older `SingleTurnSample`-style metric examples as legacy and recommend the collections API for new projects.

# Final Architecture

```text
Evaluation Dataset
      ↓
    Query
      ↓
     RAG
  ┌───┴────┐
  │        │
Retriever Generator
  │        │
Contexts Response
  └───┬────┘
      ↓
    RAGAS
  ├─ Context Precision
  ├─ Context Recall
  ├─ Faithfulness
  ├─ Answer Relevancy
  └─ Factual Correctness
      ↓
 Evaluation Report
```